# 03 - Baseline: Popularidade Global

Recomenda os produtos mais comprados globalmente, ignorando qualquer personalização. É o piso de comparação: qualquer modelo mais sofisticado deve superá-lo. Ver `docs/NOTEBOOKS.md` (seção 3).

## 0. Configuração Inicial

In [1]:
import json
import pickle
import random
import sys
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
import scipy.sparse as sp
import yaml

sys.path.insert(0, str(Path("..").resolve()))
from src.evaluation.metrics import evaluate_recommendations, pairs_to_ground_truth

RANDOM_SEED = 42


def set_seed(seed: int) -> None:
    """Fixa a seed de aleatoriedade para reprodutibilidade."""
    random.seed(seed)
    np.random.seed(seed)


set_seed(RANDOM_SEED)

PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models/baseline_popularity")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

with open("../configs/model_config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)
K = config["evaluation"]["k"]

mlflow.set_tracking_uri(f"file:{(Path('..') / 'mlruns').resolve()}")
mlflow.set_experiment("baseline_popularity")

print(f"K={K}")

K=10


## 1. Carregamento dos Artefatos de Preprocessing

In [2]:
with open(PROCESSED_DIR / "vocabularies.pkl", "rb") as f:
    vocab = pickle.load(f)

interactions_prior = sp.load_npz(PROCESSED_DIR / "interactions_prior.npz")
val_pairs = pd.read_pickle(PROCESSED_DIR / "val_pairs.pkl")
test_pairs = pd.read_pickle(PROCESSED_DIR / "test_pairs.pkl")

n_users, n_items = interactions_prior.shape
print(f"n_users={n_users:,} | n_items={n_items:,}")

n_users=126,408 | n_items=3,000


## 2. Ranking Global de Popularidade

In [3]:
item_frequency = np.asarray(interactions_prior.sum(axis=0)).flatten()
global_ranking = np.argsort(-item_frequency)

top_10_names = vocab["idx_to_product_name"][global_ranking[:10]]
list(top_10_names)

['Banana',
 'Bag of Organic Bananas',
 'Organic Strawberries',
 'Organic Baby Spinach',
 'Large Lemon',
 'Limes',
 'Organic Hass Avocado',
 'Strawberries',
 'Organic Avocado',
 'Organic Blueberries']

## 3. Geração de Recomendações (Mesma Lista para Todos os Usuários)

In [4]:
def build_popularity_recommendations(
    users: list[int], ranking: np.ndarray, k: int
) -> dict[int, list[int]]:
    """Gera o mesmo top-k global de itens populares para cada usuário.

    Args:
        users: Lista de user_idx avaliados.
        ranking: Itens ordenados por popularidade decrescente.
        k: Tamanho do top-k recomendado.

    Returns:
        Mapeamento user_idx -> lista de item_idx recomendados.
    """
    top_k = ranking[:k].tolist()
    return {user_idx: top_k for user_idx in users}


val_ground_truth = pairs_to_ground_truth(val_pairs)
test_ground_truth = pairs_to_ground_truth(test_pairs)

val_recommendations = build_popularity_recommendations(
    list(val_ground_truth.keys()), global_ranking, K
)
test_recommendations = build_popularity_recommendations(
    list(test_ground_truth.keys()), global_ranking, K
)

## 4. Avaliação

In [5]:
val_metrics = evaluate_recommendations(val_recommendations, val_ground_truth, K)
test_metrics = evaluate_recommendations(test_recommendations, test_ground_truth, K)

print("Validação:", val_metrics)
print("Teste:", test_metrics)

Validação: {'precision_at_k': 0.07621960867042878, 'recall_at_k': 0.09450904134471746, 'ndcg_at_k': 0.11184213218083841, 'map_at_k': 0.05347586449644308}
Teste: {'precision_at_k': 0.07473367788207995, 'recall_at_k': 0.0935275059068729, 'ndcg_at_k': 0.10969019009039455, 'map_at_k': 0.05221464005008067}


## 5. Persistência e Rastreamento no MLflow

In [6]:
with open(PROCESSED_DIR / "split_meta.json", encoding="utf-8") as f:
    split_meta = json.load(f)

with open(MODELS_DIR / "ranking.pkl", "wb") as f:
    pickle.dump(global_ranking, f)

metrics_payload = {"k": K, "validation": val_metrics, "test": test_metrics}
with open(MODELS_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

with mlflow.start_run(run_name="baseline_popularity_v1"):
    mlflow.log_param("k", K)
    mlflow.log_param("dataset_hash", split_meta["dataset_hash"])
    mlflow.log_metrics({f"val_{name}": value for name, value in val_metrics.items()})
    mlflow.log_metrics({f"test_{name}": value for name, value in test_metrics.items()})
    mlflow.log_artifact(str(MODELS_DIR / "metrics.json"))

print("Baseline de popularidade avaliado e rastreado no MLflow.")

Baseline de popularidade avaliado e rastreado no MLflow.
